<a href="https://colab.research.google.com/github/Pulakspbl/Machine-learning-in-Python-with-scikit-learn/blob/main/Module_6_Ensemble%20Model/Module_6_Ensemble_Model_Exercise_M6_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📝 Exercise M6.01

The aim of this notebook is to investigate if we can tune the hyperparameters
of a bagging regressor and evaluate the gain obtained.

We will load the California housing dataset and split it into a training and a
testing set.

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

data, target = fetch_california_housing(as_frame=True, return_X_y=True)
target *= 100  # rescale the target in k$
data_train, data_test, target_train, target_test = train_test_split(
    data, target, random_state=0, test_size=0.5
)

Create a BaggingRegressor and provide a DecisionTreeRegressor to its parameter estimator. Train the regressor and evaluate its generalization performance on the testing set using the mean absolute error.

In [3]:
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

bagg_tree = BaggingRegressor(
    estimator= DecisionTreeRegressor(max_depth=3),
    n_estimators=100,
    random_state=0
            )

bagg_tree.fit(data_train, target_train)
y_pred = bagg_tree.predict(data_test)

mae = mean_absolute_error(y_pred,target_test)
print(f"mean absolute error: {mae :.3f}")

mean absolute error: 57.550


Now, create a RandomizedSearchCV instance using the previous model and tune the important parameters of the bagging regressor. Find the best parameters and check if you are able to find a set of parameters that improve the default regressor still using the mean absolute error as a metric.

In [4]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_samples": [0.5, 0.7, 1.0],
    "max_features": [0.5, 0.7, 1.0],
    "estimator__max_depth": [2, 3, 5, 10, None]
}

random_search = RandomizedSearchCV(
    bagg_tree,
    param_distributions=param_grid,
    n_iter=20,
    scoring="neg_mean_absolute_error",
    cv=5,
    random_state=0,
    n_jobs=-1
)

random_search.fit(data_train, target_train)

RandomizedSearchCV(cv=5,
                   estimator=BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=3),
                                              n_estimators=100,
                                              random_state=0),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'estimator__max_depth': [2, 3, 5, 10,
                                                                 None],
                                        'max_features': [0.5, 0.7, 1.0],
                                        'max_samples': [0.5, 0.7, 1.0],
                                        'n_estimators': [50, 100, 200]},
                   random_state=0, scoring='neg_mean_absolute_error')

In [5]:
print(random_search.best_params_)
print(-random_search.best_score_)

{'n_estimators': 200, 'max_samples': 0.7, 'max_features': 1.0, 'estimator__max_depth': None}
35.22172757122092


In [6]:
#Now check the test-set performance
y_pred = random_search.predict(data_test)

mae = mean_absolute_error(target_test, y_pred)

print(f"Test MAE: {mae:.2f}")

Test MAE: 34.86
